# CatBoost same-budget benchmark for QLoRA comparison

Цель ноутбука - проверить, насколько честно сравнивать LLM с CatBoost, если CatBoost обучался на большем объёме данных.

Здесь CatBoost обучается на таких же объёмах, как QLoRA:

```text
20k train = 10k Yes + 10k No
30k train = 15k Yes + 15k No
val/test = 2k examples = 1k Yes + 1k No
```

Важно: popularity-признаки считаются только по выбранному train subset. Для train используется out-of-fold схема, для val/test - только информация из train subset.


In [ ]:
# Optional dependency installation:
# !pip install -q catboost pandas pyarrow scikit-learn


In [ ]:
import os, json, shutil, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, precision_score, recall_score, f1_score

SEED = 42
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path("F:/Coursework")

print("PROJECT_ROOT:", PROJECT_ROOT)

## 1. Configuration

In [ ]:
LABEL_COL = "label_strict"
ITEM_COL = "target_app_id"
ALPHA = 10.0

# Сравниваем два бюджета обучения:
# 20k total = 10k per class
# 30k total = 15k per class
TRAIN_BUDGETS_PER_CLASS = [10000, 15000]

# Такой же eval-size, как в основном QLoRA eval:
EVAL_PER_CLASS = 1000

OUT_DIR = PROJECT_ROOT / "outputs" / "baselines" / "same_budget"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = PROJECT_ROOT / "data" / "final" / "tabular_temporal"
INSTRUCTION_DIR = PROJECT_ROOT / "data" / "final" / "instruction_temporal"
SEMANTIC_DIR = PROJECT_ROOT / "outputs" / "semantic"
SEMANTIC_ZIP = SEMANTIC_DIR / "steam_semantic_outputs.zip"

print("OUT_DIR:", OUT_DIR)
print("DATA_DIR:", DATA_DIR)
print("INSTRUCTION_DIR:", INSTRUCTION_DIR)
print("SEMANTIC_DIR:", SEMANTIC_DIR)

## 2. Paths check

In [ ]:
paths = {
    "train_tabular": DATA_DIR / "train_tabular.parquet",
    "val_tabular": DATA_DIR / "val_tabular.parquet",
    "test_tabular": DATA_DIR / "test_tabular.parquet",
    "train_instruction": INSTRUCTION_DIR / "train.jsonl",
    "val_instruction": INSTRUCTION_DIR / "val.jsonl",
    "test_instruction": INSTRUCTION_DIR / "test.jsonl",
}

for name, path in paths.items():
    print(f"{name}: {path} -> {'OK' if path.exists() else 'MISSING'}")

print("SEMANTIC_DIR exists:", SEMANTIC_DIR.exists())
print("SEMANTIC_ZIP exists:", SEMANTIC_ZIP.exists())

## 3. Load data

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

train_tab = pd.read_parquet(paths["train_tabular"])
val_tab = pd.read_parquet(paths["val_tabular"])
test_tab = pd.read_parquet(paths["test_tabular"])

train_instr = load_jsonl(paths["train_instruction"])
val_instr = load_jsonl(paths["val_instruction"])
test_instr = load_jsonl(paths["test_instruction"])

print("tabular shapes:", train_tab.shape, val_tab.shape, test_tab.shape)
print("instruction shapes:", train_instr.shape, val_instr.shape, test_instr.shape)

for name, df in [("train_tab", train_tab), ("val_tab", val_tab), ("test_tab", test_tab)]:
    print(name, df[LABEL_COL].value_counts(dropna=False).to_dict())

for name, df in [("train_instr", train_instr), ("val_instr", val_instr), ("test_instr", test_instr)]:
    print(name, df["output"].value_counts(dropna=False).to_dict())

## 4. Build QLoRA-like balanced sample ids

In [ ]:
def class_mask_instruction(df, cls):
    return df["output"].astype(str).str.strip().str.lower() == cls.lower()

def make_balanced_instruction_sample(df, n_per_class, seed=42):
    yes = df[class_mask_instruction(df, "yes")].sample(n=n_per_class, random_state=seed)
    no = df[class_mask_instruction(df, "no")].sample(n=n_per_class, random_state=seed)
    return pd.concat([yes, no], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)

sample_id_sets = {}

for n_per_class in TRAIN_BUDGETS_PER_CLASS:
    key = f"train_{n_per_class * 2}"
    sample = make_balanced_instruction_sample(train_instr, n_per_class, SEED)
    sample_id_sets[key] = set(sample["sample_id"].astype(str))
    print(key, sample.shape, sample["output"].value_counts().to_dict())

val_eval_instr = make_balanced_instruction_sample(val_instr, EVAL_PER_CLASS, SEED)
test_eval_instr = make_balanced_instruction_sample(test_instr, EVAL_PER_CLASS, SEED)
sample_id_sets["val_eval"] = set(val_eval_instr["sample_id"].astype(str))
sample_id_sets["test_eval"] = set(test_eval_instr["sample_id"].astype(str))

print("val_eval:", val_eval_instr.shape, val_eval_instr["output"].value_counts().to_dict())
print("test_eval:", test_eval_instr.shape, test_eval_instr["output"].value_counts().to_dict())

## 5. Feature lists

Ниже используется безопасный набор признаков: признаки истории, пересечения тегов/жанров и базовые признаки игры.  
Глобальные агрегаты вроде `positive_ratio`, `user_reviews`, `recommendations`, `peak_ccu` не используются в основной модели, чтобы не смешивать честный popularity-признак с внешними агрегатами.


In [ ]:
SAFE_FEATURES = [
    "history_len",
    "history_positive_count",
    "history_negative_count",
    "history_positive_share",
    "history_total_hours",
    "history_mean_hours",
    "history_median_hours",
    "history_max_hours",
    "history_min_hours",
    "history_liked_mean_hours",
    "history_disliked_mean_hours",
    "target_token_count",
    "liked_token_count",
    "disliked_token_count",
    "target_liked_overlap_count",
    "target_disliked_overlap_count",
    "target_liked_jaccard",
    "target_disliked_jaccard",
    "target_jaccard_diff",
    "target_description_len",
    "target_title_len",
    "price",
    "required_age",
    "dlc_count",
    "release_year",
]

POP_FEATURES = [
    "train_item_popularity_score",
    "train_item_log_count",
]

SEMANTIC_FEATURES = [
    "semantic_target_liked_cosine",
    "semantic_target_disliked_cosine",
    "semantic_similarity_gap",
    "semantic_max_liked_cosine",
    "semantic_max_disliked_cosine",
    "semantic_top3_liked_mean_cosine",
    "semantic_top3_disliked_mean_cosine",
]

def existing_features(df, features):
    return [c for c in features if c in df.columns]

print("safe features existing:", len(existing_features(train_tab, SAFE_FEATURES)), existing_features(train_tab, SAFE_FEATURES))
print("missing safe features:", [c for c in SAFE_FEATURES if c not in train_tab.columns])

## 6. Optional semantic features

In [ ]:
def maybe_extract_semantic_zip():
    if SEMANTIC_DIR.exists():
        expected = [
            SEMANTIC_DIR / "train_semantic_features.parquet",
            SEMANTIC_DIR / "val_semantic_features.parquet",
            SEMANTIC_DIR / "test_semantic_features.parquet",
        ]
        if all(p.exists() for p in expected):
            return
    if SEMANTIC_ZIP.exists():
        SEMANTIC_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(SEMANTIC_ZIP, "r") as z:
            z.extractall(SEMANTIC_DIR)
        print("Extracted semantic zip to", SEMANTIC_DIR)

def load_semantic_split(split):
    path = SEMANTIC_DIR / f"{split}_semantic_features.parquet"
    if not path.exists():
        return None
    return pd.read_parquet(path)

maybe_extract_semantic_zip()

sem_train = load_semantic_split("train")
sem_val = load_semantic_split("val")
sem_test = load_semantic_split("test")

print("semantic loaded:", sem_train is not None, sem_val is not None, sem_test is not None)
if sem_train is not None:
    print("semantic shapes:", sem_train.shape, sem_val.shape, sem_test.shape)
    print("semantic columns:", [c for c in SEMANTIC_FEATURES if c in sem_train.columns])

def merge_semantic(tab, sem):
    if sem is None:
        return tab.copy()
    keep = ["sample_id"] + [c for c in SEMANTIC_FEATURES if c in sem.columns]
    return tab.merge(sem[keep], on="sample_id", how="left")

train_tab_sem = merge_semantic(train_tab, sem_train)
val_tab_sem = merge_semantic(val_tab, sem_val)
test_tab_sem = merge_semantic(test_tab, sem_test)

print("after semantic:", train_tab_sem.shape, val_tab_sem.shape, test_tab_sem.shape)

## 7. Train-only popularity encoding

In [ ]:
def add_train_only_popularity(train_df, val_df, test_df, item_col=ITEM_COL, label_col=LABEL_COL, alpha=ALPHA, seed=SEED, n_splits=5):
    train_df = train_df.copy()
    val_df = val_df.copy()
    test_df = test_df.copy()

    global_rate = float(train_df[label_col].mean())

    train_df["train_item_popularity_score"] = global_rate
    train_df["train_item_log_count"] = 0.0

    y = train_df[label_col].astype(int).values
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for fold, (fit_idx, enc_idx) in enumerate(skf.split(train_df, y), start=1):
        fit_part = train_df.iloc[fit_idx]
        stats = (
            fit_part.groupby(item_col)[label_col]
            .agg(["sum", "count"])
            .rename(columns={"sum": "pos_sum", "count": "cnt"})
        )
        stats["train_item_popularity_score"] = (stats["pos_sum"] + alpha * global_rate) / (stats["cnt"] + alpha)
        stats["train_item_log_count"] = np.log1p(stats["cnt"])

        enc_items = train_df.iloc[enc_idx][[item_col]].merge(
            stats[["train_item_popularity_score", "train_item_log_count"]],
            left_on=item_col,
            right_index=True,
            how="left"
        )

        train_df.iloc[enc_idx, train_df.columns.get_loc("train_item_popularity_score")] = (
            enc_items["train_item_popularity_score"].fillna(global_rate).values
        )
        train_df.iloc[enc_idx, train_df.columns.get_loc("train_item_log_count")] = (
            enc_items["train_item_log_count"].fillna(0.0).values
        )

    full_stats = (
        train_df.groupby(item_col)[label_col]
        .agg(["sum", "count"])
        .rename(columns={"sum": "pos_sum", "count": "cnt"})
    )
    full_stats["train_item_popularity_score"] = (full_stats["pos_sum"] + alpha * global_rate) / (full_stats["cnt"] + alpha)
    full_stats["train_item_log_count"] = np.log1p(full_stats["cnt"])

    def apply_to_eval(df):
        out = df.merge(
            full_stats[["train_item_popularity_score", "train_item_log_count"]],
            left_on=item_col,
            right_index=True,
            how="left"
        )
        out["train_item_popularity_score"] = out["train_item_popularity_score"].fillna(global_rate)
        out["train_item_log_count"] = out["train_item_log_count"].fillna(0.0)
        return out

    val_df = apply_to_eval(val_df)
    test_df = apply_to_eval(test_df)

    return train_df, val_df, test_df

## 8. Model and metrics

In [ ]:
def best_f1_threshold(y_true, scores):
    thresholds = np.linspace(0.01, 0.99, 99)
    best_t, best_f1 = 0.5, -1.0
    for t in thresholds:
        pred = (scores >= t).astype(int)
        cur = f1_score(y_true, pred, zero_division=0)
        if cur > best_f1:
            best_t, best_f1 = float(t), float(cur)
    return best_t, best_f1

def compute_metrics(y_true, scores, threshold):
    pred = (scores >= threshold).astype(int)
    return {
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "pr_auc": float(average_precision_score(y_true, scores)),
        "accuracy": float(accuracy_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "threshold": float(threshold),
    }

def fit_predict_catboost(train_df, val_df, test_df, features, seed=SEED):
    features = existing_features(train_df, features)
    if not features:
        raise ValueError("No features found")

    X_train = train_df[features].copy()
    y_train = train_df[LABEL_COL].astype(int).values
    X_val = val_df[features].copy()
    y_val = val_df[LABEL_COL].astype(int).values
    X_test = test_df[features].copy()
    y_test = test_df[LABEL_COL].astype(int).values

    model = CatBoostClassifier(
        iterations=700,
        depth=6,
        learning_rate=0.05,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=seed,
        verbose=False,
        allow_writing_files=False
    )

    model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)
    val_scores = model.predict_proba(X_val)[:, 1]
    test_scores = model.predict_proba(X_test)[:, 1]

    threshold, _ = best_f1_threshold(y_val, val_scores)
    val_metrics = compute_metrics(y_val, val_scores, threshold)
    test_metrics = compute_metrics(y_test, test_scores, threshold)

    val_pred = pd.DataFrame({
        "sample_id": val_df["sample_id"].astype(str).values,
        "y_true": y_val,
        "score": val_scores,
        "pred": (val_scores >= threshold).astype(int)
    })
    test_pred = pd.DataFrame({
        "sample_id": test_df["sample_id"].astype(str).values,
        "y_true": y_test,
        "score": test_scores,
        "pred": (test_scores >= threshold).astype(int)
    })

    return model, val_metrics, test_metrics, val_pred, test_pred, features

## 9. Run benchmarks

In [ ]:
results = []
all_predictions = {}

val_eval = val_tab_sem[val_tab_sem["sample_id"].astype(str).isin(sample_id_sets["val_eval"])].copy()
test_eval = test_tab_sem[test_tab_sem["sample_id"].astype(str).isin(sample_id_sets["test_eval"])].copy()

print("val_eval:", val_eval.shape, val_eval[LABEL_COL].value_counts().to_dict())
print("test_eval:", test_eval.shape, test_eval[LABEL_COL].value_counts().to_dict())

if len(val_eval) != EVAL_PER_CLASS * 2 or len(test_eval) != EVAL_PER_CLASS * 2:
    raise ValueError("Eval subset size mismatch. Check sample_id alignment.")

model_specs = [
    ("safe_features", SAFE_FEATURES),
    ("safe_plus_train_popularity", SAFE_FEATURES + POP_FEATURES),
]

if all(c in train_tab_sem.columns for c in SEMANTIC_FEATURES):
    model_specs.append(("safe_plus_train_popularity_plus_semantic", SAFE_FEATURES + POP_FEATURES + SEMANTIC_FEATURES))
else:
    print("Semantic features are not fully available. Semantic model will be skipped.")

for n_per_class in TRAIN_BUDGETS_PER_CLASS:
    budget_name = f"train_{n_per_class * 2}"
    train_budget = train_tab_sem[train_tab_sem["sample_id"].astype(str).isin(sample_id_sets[budget_name])].copy()

    print("\n====", budget_name, "====")
    print("train_budget:", train_budget.shape, train_budget[LABEL_COL].value_counts().to_dict())

    if len(train_budget) != n_per_class * 2:
        raise ValueError(f"Train subset size mismatch for {budget_name}")

    train_pop, val_pop, test_pop = add_train_only_popularity(train_budget, val_eval, test_eval)

    for model_name, features in model_specs:
        print("Training:", budget_name, model_name)
        model, val_metrics, test_metrics, val_pred, test_pred, used_features = fit_predict_catboost(
            train_pop, val_pop, test_pop, features
        )

        row_val = {
            "budget": budget_name,
            "train_examples": int(len(train_pop)),
            "eval_examples": int(len(val_pop)),
            "model": model_name,
            "split": "val",
            "n_features": int(len(used_features)),
            **val_metrics
        }
        row_test = {
            "budget": budget_name,
            "train_examples": int(len(train_pop)),
            "eval_examples": int(len(test_pop)),
            "model": model_name,
            "split": "test",
            "n_features": int(len(used_features)),
            **test_metrics
        }
        results.extend([row_val, row_test])

        pred_key = f"{budget_name}_{model_name}"
        all_predictions[(pred_key, "val")] = val_pred
        all_predictions[(pred_key, "test")] = test_pred

        print("VAL:", val_metrics)
        print("TEST:", test_metrics)

metrics_df = pd.DataFrame(results)
metrics_df.sort_values(["budget", "model", "split"])

## 10. Save outputs

In [ ]:
metrics_path = OUT_DIR / "catboost_same_budget_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)

for (pred_key, split), pred_df in all_predictions.items():
    pred_df.to_csv(OUT_DIR / f"{pred_key}_{split}_predictions.csv", index=False)

summary = {
    "seed": int(SEED),
    "train_budgets_per_class": [int(x) for x in TRAIN_BUDGETS_PER_CLASS],
    "eval_per_class": int(EVAL_PER_CLASS),
    "label_col": LABEL_COL,
    "item_col": ITEM_COL,
    "alpha": float(ALPHA),
    "metrics": metrics_df.to_dict(orient="records"),
    "notes": [
        "Train subsets are sampled from instruction_temporal with the same random_state=42 style as QLoRA notebooks.",
        "Validation and test subsets are balanced 1000 Yes + 1000 No, matching QLoRA eval-only protocol.",
        "Popularity features are computed only from the selected train subset.",
        "For train, popularity features use out-of-fold encoding."
    ]
}

summary_path = OUT_DIR / "catboost_same_budget_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

archive_path = shutil.make_archive(str(OUT_DIR / "catboost_same_budget_outputs"), "zip", OUT_DIR)

print("Saved metrics:", metrics_path)
print("Saved summary:", summary_path)
print("Archive:", archive_path)

metrics_df

## Output artifacts

Main run artifacts are written to:

```text
outputs/baselines/same_budget/catboost_same_budget_outputs.zip
```

The archive contains metrics, predictions and a summary file.
